In [1]:
import pandas as pd

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix


In [3]:
df=pd.read_csv("diabetes.csv")
print(df.head())

   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [5]:
mm=MinMaxScaler()
df['Glucose']=mm.fit_transform(df[['Glucose']])
df['BMI']=mm.fit_transform(df[["BMI"]])
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,0.743719,72,35,0,0.500745,0.627,50,1
1,1,0.427136,66,29,0,0.396423,0.351,31,0
2,8,0.919598,64,0,0,0.347243,0.672,32,1
3,1,0.447236,66,23,94,0.418778,0.167,21,0
4,0,0.688442,40,35,168,0.642325,2.288,33,1


In [6]:
x=df.drop("Outcome",axis=1)
y=df["Outcome"]

In [7]:
print(x.shape,y.shape)

(768, 8) (768,)


In [8]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [9]:
print(x_train.shape,x_test.shape,y_train.shape,y_test.shape)


(614, 8) (154, 8) (614,) (154,)


In [10]:
clf=SVC()
param_grid={
    'kernel':['Linear','rbf'],
    'C':[0.001,0.01,0.1,1,10,100],
    'gamma':['scale','auto']
}

In [11]:
grid=GridSearchCV(SVC(),param_grid,cv=5,scoring='accuracy')
grid.fit(x_train,y_train)
result=pd.DataFrame(grid.cv_results_)
print(result[[
    'params',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
]])

print("\n Best Parameters: ")
print(grid.best_params_)

                                               params  mean_test_score  \
0   {'C': 0.001, 'gamma': 'scale', 'kernel': 'Line...              NaN   
1     {'C': 0.001, 'gamma': 'scale', 'kernel': 'rbf'}         0.651473   
2   {'C': 0.001, 'gamma': 'auto', 'kernel': 'Linear'}              NaN   
3      {'C': 0.001, 'gamma': 'auto', 'kernel': 'rbf'}         0.651473   
4   {'C': 0.01, 'gamma': 'scale', 'kernel': 'Linear'}              NaN   
5      {'C': 0.01, 'gamma': 'scale', 'kernel': 'rbf'}         0.651473   
6    {'C': 0.01, 'gamma': 'auto', 'kernel': 'Linear'}              NaN   
7       {'C': 0.01, 'gamma': 'auto', 'kernel': 'rbf'}         0.651473   
8    {'C': 0.1, 'gamma': 'scale', 'kernel': 'Linear'}              NaN   
9       {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}         0.649847   
10    {'C': 0.1, 'gamma': 'auto', 'kernel': 'Linear'}              NaN   
11       {'C': 0.1, 'gamma': 'auto', 'kernel': 'rbf'}         0.651473   
12     {'C': 1, 'gamma': 'scale', 'ker

c:\Users\shubh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
60 fits failed out of a total of 120.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
60 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\shubh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\shubh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py", line 1382, in wrapper
    estimator._validate_params()
  File "c:\Users\shubh\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.p

In [12]:
best_model=grid.best_estimator_
ypred_acc=best_model.predict(x_test)

In [13]:
print("\n Accuracy Score: ")
print(accuracy_score(y_test,ypred_acc))

print("\n Classification Reports: ")
print(classification_report(y_test,ypred_acc))

print("\n Confusion Matrix")
print(confusion_matrix(y_test,ypred_acc))


 Accuracy Score: 
0.6753246753246753

 Classification Reports: 
              precision    recall  f1-score   support

           0       0.68      0.95      0.79       100
           1       0.64      0.17      0.26        54

    accuracy                           0.68       154
   macro avg       0.66      0.56      0.53       154
weighted avg       0.67      0.68      0.61       154


 Confusion Matrix
[[95  5]
 [45  9]]
